## Data cleaning and Preprocessing

In [ ]:
# adding necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from scipy.stats import chi2_contingency
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings

In [ ]:
# opening the dataset
data_filepath = "train_data/group_01_train.csv"
data = pd.read_csv(data_filepath)
data

In [ ]:
data.dtypes.unique()

In [ ]:
# dropping f11 because the only value used in this column is is 'US'
data = data.drop(columns=['f11'])

In [ ]:
# dropping f7 because converting it to a numerical value would need LLMs and NLPs which are not discussed in this course
data = data.drop(columns=['f7'])

In [ ]:
# dropping f8 because of the same reason for the above cell
data = data.drop(columns=['f8'])

In [ ]:
pd.DataFrame({
    'Dtype': data.dtypes,
    'Missing Count': data.isna().sum()
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

In [ ]:
data.info()

In [ ]:
data.nunique()

In [ ]:
# we draw boxplot of dutation between f2 and f13 (f2 - f13) and see if there is any
# meaningful relation between y and this feature.

data['f13'] = pd.to_datetime(data['f13'], format='mixed')
data['f2'] = pd.to_datetime(data['f2'], format='mixed')

data['duration_min'] = (data['f2'] - data['f13']).dt.total_seconds() / 60.0

df_clean = data[data['duration_min'] > 0].copy()

sns.boxplot(data=df_clean, x='y', y='duration_min', showfliers=False)
plt.title('Event Duration vs Target Label (y)')
plt.xlabel('Target Label (y)')
plt.ylabel('Duration (Minutes)')
plt.show()

In [ ]:
# we use f13 to extract hour, day of the week, and season of the record and save them in different columns.

data['f13'] = pd.to_datetime(data['f13'], format='mixed')

def get_season(month):
    if month in [12, 1, 2]:
        return 4
    elif month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    else:
        return 3
    
data['hour'] = data['f13'].dt.hour
data['day_of_week'] = data['f13'].dt.dayofweek
data['season'] = data['f13'].dt.month.apply(get_season)

In [ ]:
# we show the proportion of each target within each season and see if there is any
# meaningful relation between them.
pd.crosstab(data['season'], data['y'], normalize='index')

In [ ]:
# we draw a histogram to see relationship between hour, and 4 columns which are Day/Night (f31, f32, f33, f34)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
cols = ['f31', 'f32', 'f33', 'f34']
axes_flat = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(
        data=data, 
        x='hour', 
        hue=col, 
        multiple='dodge', 
        bins=24, 
        shrink=0.8, 
        ax=axes_flat[i]
    )
    axes_flat[i].set_title(f'Relationship Between Extracted Hour and Column {col}')
    axes_flat[i].set_xlabel('Hour of the Day (0-23)')
    axes_flat[i].set_ylabel('Accident Count')
    axes_flat[i].set_xticks(range(0, 24))


plt.tight_layout()
fig.savefig('hour_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# we drop these 4 columns becuase they show high colinearity with `hour`
data = data.drop(columns=['f31', 'f32', 'f33', 'f34'])

In [ ]:
# we drop these f2 and f13 cause their information is extracted and they're not needed anymore.
data = data.drop(columns=['f2', 'f13'])
data.columns

In [ ]:
# Dropping duplicates from the training set
print("Training rows before deduplication:", len(data))
data = data.drop_duplicates()
print("Training rows after deduplication:", len(data))

In [ ]:
# splitting the dataset into 15% test, 15% validation and 70% training sets

X = data.drop(columns=['y'])
y = data['y']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=123, stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=123, stratify=y_temp
)

print(f"Training set:   {X_train.shape}, Labels: {y_train.shape}")
print(f"Validation set: {X_val.shape}, Labels: {y_val.shape}")
print(f"Test set:       {X_test.shape}, Labels: {y_test.shape}")

In [ ]:
alpha = 0.05
missing_percentage = X_train.isnull().mean() * 100
over_miss_cols = missing_percentage[missing_percentage > 25].index

print("--- Checking Missingness vs Target 'y' (Chi-Square Test) ---")

cols_to_drop = []
cols_to_keep_and_impute = {}

for col in over_miss_cols:
    missing_indicator_train = X_train[col].isna().astype(int)
    contingency_table = pd.crosstab(missing_indicator_train, y_train)

    contingency_table = pd.crosstab(
        missing_indicator_train,
        y_train,
        rownames=["Is_Missing"],
        colnames=["Target_y"],
    )

    print()
    print("=" * 50)
    print(f"Column: '{col}'")
    print("-" * 50)
    print("Contingency Table (Counts):")
    print(contingency_table)
    print("-" * 50)

    # ensure at least one observation exists in each (missing status, target class) 
    if (contingency_table == 0).any().any() :
        print(f"Column: '{col}' - Not enough data for Chi-Square test. Skipping test.")
        # we'll mark it to be dropped if we can't test its MNAR nature
        cols_to_drop.append(col)
        continue

    chi2, p_value, _, _ = chi2_contingency(contingency_table)

    print(f"Column: '{col}' | p-value: {p_value:.5f}")

    if p_value < alpha:
        print(
            f"Missingness is RELATED to 'y' (p < {alpha}). Keeping the column and impuation info for later."
        )
        # Impute missing values:
        if pd.api.types.is_numeric_dtype(X_train[col]):
            fill_val = X_train[col].median()
            print(
                f"-> New indicator column '{col}_is_missing' and imputed '{col}' with median ('{fill_val}')."
            )
        else:
            fill_val = "[MISSING]"
            print(
                f"-> New indicator column '{col}_is_missing' and imputed '{col}' with new category '{fill_val}'."
            )

        cols_to_keep_and_impute[col] = fill_val

    else:
        print(
            f"-> Missingness is NOT related to 'y' (p >= {alpha}). Marking column for dropping."
        )
        cols_to_drop.append(col)

for df in [X_train, X_val, X_test]:
    for col, fill_val in cols_to_keep_and_impute.items():
        df[f"{col}_is_missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(fill_val)
    
    df.drop(columns=cols_to_drop, inplace=True)


In [ ]:
remaining_missing = X_train.isnull().mean() * 100
cols_to_impute = remaining_missing[remaining_missing > 0].index

num_cols = X_train[cols_to_impute].select_dtypes(include='number').columns
if not num_cols.empty:
    num_imputer = SimpleImputer(strategy='mean')
    X_train.loc[:, num_cols] = num_imputer.fit_transform(X_train[num_cols])
    X_val.loc[:, num_cols] = num_imputer.transform(X_val[num_cols])
    X_test.loc[:, num_cols] = num_imputer.transform(X_test[num_cols])

    for col, mean_val in zip(num_cols, num_imputer.statistics_):
        print(f"Imputating numerical column {col} with {mean_val}")


cat_cols = X_train[cols_to_impute].select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    fill_label = "Unknown"
    existing_values = X_train[col].dropna().unique()
    if fill_label in existing_values:
        i = 1
        while f"Unknown_{i}" in existing_values:
            i += 1
        fill_label = f"Unknown_{i}"

    print(f"Imputating string column {col} with {fill_label}")
    X_train[col] = X_train[col].fillna(fill_label)
    X_val[col] = X_val[col].fillna(fill_label)
    X_test[col] = X_test[col].fillna(fill_label)


In [ ]:
# mean imputer for numerical numbers on the trainig dataset, as well as for validation and test separately

numeric_cols = X_train.select_dtypes(include='number').columns
imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train[numeric_cols])

X_train.loc[:, numeric_cols] = imputer.transform(X_train[numeric_cols])
X_val.loc[:, numeric_cols] = imputer.transform(X_val[numeric_cols])
X_test.loc[:, numeric_cols] = imputer.transform(X_test[numeric_cols])

In [ ]:
# frequency encoding for text dtype high cardinality columns
cols = ['f9', 'f10', 'f18', 'f21']

for col in cols:
    freq = X_train[col].value_counts()
    
    X_train[col] = X_train[col].map(freq).astype(int)
    X_val[col] = X_val[col].map(freq).fillna(0).astype(int)
    X_test[col] = X_test[col].map(freq).fillna(0).astype(int)
    
    print(f"Frequency encoded {col}")


In [ ]:
# ont-hot-oncoding for text dtype categorical columns
cols = ['f1', 'f12']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=int)

train_encoded = ohe.fit_transform(X_train[cols])
val_encoded = ohe.transform(X_val[cols])
test_encoded = ohe.transform(X_test[cols])

feature_names = ohe.get_feature_names_out(cols)
train_encoded_df = pd.DataFrame(train_encoded, columns=feature_names, index=X_train.index)
val_encoded_df = pd.DataFrame(val_encoded, columns=feature_names, index=X_val.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=feature_names, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=cols), train_encoded_df], axis=1)
X_val = pd.concat([X_val.drop(columns=cols), val_encoded_df], axis=1)
X_test = pd.concat([X_test.drop(columns=cols), test_encoded_df], axis=1)

In [ ]:
# Converting boolean columns to integers for ML compatibility
bool_cols = X_train.select_dtypes(include='bool').columns

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_val[bool_cols] = X_val[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [ ]:
# columns standardization
num_cols = X_train.select_dtypes(include='number').columns

X_train[num_cols] = X_train[num_cols].astype(float)
X_val[num_cols] = X_val[num_cols].astype(float)
X_test[num_cols] = X_test[num_cols].astype(float)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols]).round(2)
X_val[num_cols] = scaler.transform(X_val[num_cols]).round(2)
X_test[num_cols] = scaler.transform(X_test[num_cols]).round(2)

In [ ]:
# Tree-Based RFC Feature selection
rf_model = RandomForestClassifier(n_estimators=100, random_state=123, n_jobs=-1)

selector = SelectFromModel(estimator=rf_model, max_features=25, threshold=-np.inf)

selector.fit(X_train, y_train)

selected_cols = X_train.columns[selector.get_support()]
dropped_cols = [col for col in X_train.columns if col not in selected_cols]

X_train_selected = selector.transform(X_train)
X_val_selected = selector.transform(X_val)
X_test_selected = selector.transform(X_test)

X_train = pd.DataFrame(X_train_selected, columns=selected_cols, index=X_train.index)
X_val = pd.DataFrame(X_val_selected, columns=selected_cols, index=X_val.index)
X_test = pd.DataFrame(X_test_selected, columns=selected_cols, index=X_test.index)

print(f"Features reduced from 31 to {X_train.shape[1]}")
print(f"Dropped features: {dropped_cols}")

In [ ]:
# important features visualized by our FS method
trained_rf = selector.estimator_
importances = trained_rf.feature_importances_
original_features = selector.feature_names_in_

importance_df = pd.DataFrame({
    'Feature': original_features,
    'Importance': importances
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df['Status'] = ['Kept' if feat in selected_cols else 'Dropped' for feat in importance_df['Feature']]

plt.figure(figsize=(12, 10))
sns.barplot(
    data=importance_df, 
    x='Importance', 
    y='Feature', 
    hue='Status',
    dodge=False,
    palette={'Kept': '#2ecc71', 'Dropped': '#e74c3c'} 
)

plt.title('Random Forest Feature Importances', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score (Mean Decrease in Impurity)', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.legend(title='Selection Status', loc='lower right', fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
train_final = pd.concat([X_train, y_train], axis=1)
val_final = pd.concat([X_val, y_val], axis=1)
test_final = pd.concat([X_test, y_test], axis=1)

print("\n--- Final Dataset Shapes ---")
print(f"Training Data:   {train_final.shape}")
print(f"Validation Data: {val_final.shape}")
print(f"Test Data:       {test_final.shape}")

train_final.to_csv('train_data/train_preprocessed.csv', index=False)
val_final.to_csv('train_data/val_preprocessed.csv', index=False)
test_final.to_csv('train_data/test_preprocessed.csv', index=False)

In [ ]:
X_train = X_train.astype(float)
X_val = X_val.astype(float)
X_test = X_test.astype(float)
warnings.filterwarnings("ignore")

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)
y_test_encoded = le.transform(y_test)

# Logistic Regression 
logreg_params = {
    'penalty': 'l2',              
    'C': 1.0,                    
    'solver': 'lbfgs',           
    'max_iter': 1000,           
    'class_weight': 'balanced',   
    'random_state': 123
}

# XGBoost Architecture
xgb_params = {
    'n_estimators': 200,        
    'learning_rate': 0.1,        
    'max_depth': 6,              
    'subsample': 0.8,             
    'colsample_bytree': 0.8,      
    'objective': 'multi:softmax', 
    'eval_metric': 'mlogloss',   
    'random_state': 123
}

# Multi-Layer Perceptron Architecture
mlp_params = {
    'hidden_layer_sizes': (128, 64), 
    'activation': 'relu',            
    'solver': 'adam',                
    'alpha': 0.0001,                
    'learning_rate_init': 0.001,     
    'batch_size': 'auto',           
    'max_iter': 1000,               
    'random_state': 123
}

models = {
    "Multinomial Logistic Regression": LogisticRegression(**logreg_params),
    "XGBoost": XGBClassifier(**xgb_params),
    "Multi-Layer Perceptron": MLPClassifier(**mlp_params)
}

performance_metrics = []

print("Starting Model Training & Evaluation...")
print()

for model_name, model in models.items():
    print(f"Training {model_name}...")
    
    model.fit(X_train, y_train_encoded)

    y_val_pred = model.predict(X_val)
    
    accuracy = accuracy_score(y_val_encoded, y_val_pred)
    f1 = f1_score(y_val_encoded, y_val_pred, average='weighted')
    
    performance_metrics.append({
        "Model": model_name,
        "Validation Accuracy": accuracy,
        "Validation F1-Score": f1
    })

comparison_df = pd.DataFrame(performance_metrics).sort_values(by="Validation F1-Score", ascending=False)

print()
print("="*50)
print("          MODEL COMPARISON LEADERBOARD")
print("="*50)
print(comparison_df.to_string(index=False))